In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [7]:
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"    
cpi_df = pd.read_excel(DATA_DIR / "hm1.xlsx", skiprows=[1,2,3,4], index_col=0, parse_dates=True)

In [ ]:
cpi_df = cpi_df.rename(columns={"Consumers price index (CPI)":  "cpi"})

In [29]:
cpi_df["time"] = pd.to_datetime(cpi_df["index"], format="%Y-%m")
cpi_df = cpi_df.set_index("time").drop(columns=["index"])

In [30]:
cpi_df.head()

,cpi,Consumers price index (CPI).1,Consumers price index (CPI).2,Tradable inflation,Tradable inflation.1,Tradable inflation.2,Non-tradable inflation,Non-tradable inflation.1,Non-tradable inflation.2,Food,...,PropertyIQ House price index (HPI).2,GDP deflator,GDP deflator.1,GDP deflator.2,Producers price index (PPI) inputs,Producers price index (PPI) inputs.1,Producers price index (PPI) inputs.2,Producers price index (PPI) outputs,Producers price index (PPI) outputs.1,Producers price index (PPI) outputs.2
time,,,,,,,,,,,,,,,,,,,,,
1988-03-31,516,1.8,9.0,NaN,NaN,NaN,NaN,NaN,NaN,497,...,NaN,626,2.5,NaN,529,1.0,5.7,546,0.6,5.9
1988-06-30,520,0.8,6.3,NaN,NaN,NaN,NaN,NaN,NaN,504,...,NaN,631,0.8,5.7,537,1.5,4.9,556,1.9,5.2
1988-09-30,524,0.9,5.6,NaN,NaN,NaN,NaN,NaN,NaN,509,...,NaN,651,3.2,8.5,545,1.5,5.0,567,1.9,5.7
1988-12-31,531,1.2,4.7,NaN,NaN,NaN,NaN,NaN,NaN,518,...,NaN,648,-0.5,6.1,552,1.2,5.3,577,1.8,6.4
1989-03-31,536,1.1,4.0,NaN,NaN,NaN,NaN,NaN,NaN,522,...,NaN,653,0.8,4.3,562,1.9,6.3,587,1.7,7.5


In [33]:
#Calculate the logarithmic quarterly growth rate of CPI
cpi_df['log_cpi'] = np.log(cpi_df["cpi"])
cpi_df['log_cpi_qr'] = cpi_df['log_cpi'] - cpi_df['log_cpi'].shift(1)
cpi_df[['log_cpi_qr']]*100

,log_cpi_qr
time,
1988-03-31,NaN
1988-06-30,0.772205
1988-09-30,0.766287
1988-12-31,1.327034
1989-03-31,0.937214
...,...
2025-06-30,0.537429
2025-09-30,0.990484
2025-12-31,0.604688


In [36]:
cpi_df['cpi_qr'] = cpi_df['cpi'].pct_change()
cpi_df[["cpi_qr"]]*100

,cpi_qr
time,
1988-03-31,NaN
1988-06-30,0.775194
1988-09-30,0.769231
1988-12-31,1.335878
1989-03-31,0.941620
...,...
2025-06-30,0.538876
2025-09-30,0.995406
2025-12-31,0.606520


In [40]:
cpi_df['log_cpi_yr'] = cpi_df['log_cpi'] - cpi_df['log_cpi'].shift(4)
cpi_df[['log_cpi_yr']]*100

,log_cpi_yr
time,
1988-03-31,NaN
1988-06-30,NaN
1988-09-30,NaN
1988-12-31,NaN
1989-03-31,3.802740
...,...
2025-06-30,2.637857
2025-09-30,3.001380
2025-12-31,3.060683


In [38]:
cpi_df['cpi_yr'] = cpi_df['cpi'].pct_change(periods=4)
cpi_df[["cpi_yr"]]*100

,cpi_yr
time,
1988-03-31,NaN
1988-06-30,NaN
1988-09-30,NaN
1988-12-31,NaN
1989-03-31,3.875969
...,...
2025-06-30,2.672956
2025-09-30,3.046875
2025-12-31,3.108003


In [44]:
##using AR(1) model to predict the CPI growth rate
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(cpi_df['log_cpi_qr'].dropna(), order=(1,0,0))
model_fit = model.fit()
print(model_fit.summary())


                               SARIMAX Results                                
Dep. Variable:             log_cpi_qr   No. Observations:                  153
Model:                 ARIMA(1, 0, 0)   Log Likelihood                 584.783
Date:                Mon, 19 Oct 2026   AIC                          -1163.566
Time:                        23:34:50   BIC                          -1154.475
Sample:                    06-30-1988   HQIC                         -1159.873
                         - 06-30-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0064      0.001      7.483      0.000       0.005       0.008
ar.L1          0.4261      0.084      5.069      0.000       0.261       0.591
sigma2      2.794e-05   2.11e-06     13.234      0.0

/Users/yuzhezhang/anaconda3/envs/VIX/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)
/Users/yuzhezhang/anaconda3/envs/VIX/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)
/Users/yuzhezhang/anaconda3/envs/VIX/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)


In [46]:
next_forecast = model_fit.forecast(steps=1)
print(f"Next quarter's forecasted log CPI growth rate: {next_forecast[0] * 100:.2f}%")

Next quarter's forecasted log CPI growth rate: 1.00%


/var/folders/x5/811605315pz3sbnmkwzjmcw00000gn/T/ipykernel_94891/3603716175.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Next quarter's forecasted log CPI growth rate: {next_forecast[0] * 100:.2f}%")


In [47]:
model_year_cpi = ARIMA(cpi_df['log_cpi_yr'].dropna(), order=(1,0,0))
model_year_cpi_fit = model_year_cpi.fit()
print(model_year_cpi_fit.summary())

                               SARIMAX Results                                
Dep. Variable:             log_cpi_yr   No. Observations:                  150
Model:                 ARIMA(1, 0, 0)   Log Likelihood                 528.504
Date:                Mon, 19 Oct 2026   AIC                          -1051.007
Time:                        23:36:24   BIC                          -1041.975
Sample:                    03-31-1989   HQIC                         -1047.338
                         - 06-30-2026                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0266      0.007      4.037      0.000       0.014       0.040
ar.L1          0.9084      0.033     27.179      0.000       0.843       0.974
sigma2      5.029e-05   4.14e-06     12.149      0.0

/Users/yuzhezhang/anaconda3/envs/VIX/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)
/Users/yuzhezhang/anaconda3/envs/VIX/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)
/Users/yuzhezhang/anaconda3/envs/VIX/lib/python3.10/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)


In [48]:
next_year_forecast = model_year_cpi_fit.forecast(steps=1)
print(f"Next year's forecasted log CPI growth rate: {next_year_forecast[0] * 100:.2f}%")

Next year's forecasted log CPI growth rate: 3.86%


/var/folders/x5/811605315pz3sbnmkwzjmcw00000gn/T/ipykernel_94891/2979449433.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Next year's forecasted log CPI growth rate: {next_year_forecast[0] * 100:.2f}%")
